# 05 — Final Model Error Analysis

Bu notebook test sonucunu detaylı analiz eder.

Amaçlar:
- Test accuracy, Top-3, Top-5 sonuçlarını özetlemek
- Confidence threshold seçmek
- En kötü class'ları bulmak
- En çok karışan class çiftlerini bulmak
- Yanlış ama yüksek confidence alan örnekleri incelemek
- Landmark extraction kalitesini kontrol etmek
- İncelenecek video listesini çıkarmak

Gerekli dosyalar:
- `/content/drive/MyDrive/sign-language-project/test_results_final/test_metrics_final.json`
- `/content/drive/MyDrive/sign-language-project/test_results_final/test_predictions_final.csv`
- `/content/drive/MyDrive/sign-language-project/test_results_final/test_per_class_accuracy_final.csv`
- `/content/drive/MyDrive/sign-language-project/test_results_final/test_top_confusions_final.csv`
- `/content/drive/MyDrive/sign-language-project/packed_landmarks/test_color_final_raw_X.npy`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import numpy as np
import pandas as pd

from IPython.display import display, HTML, Video

BASE = Path("/content/drive/MyDrive/sign-language-project")
DATASET = BASE / "dataset"
TEST_DIR = DATASET / "test"

RESULTS_DIR = BASE / "test_results_final"
PACKED_DIR = BASE / "packed_landmarks"

METRICS_PATH = RESULTS_DIR / "test_metrics_final.json"
PRED_PATH = RESULTS_DIR / "test_predictions_final.csv"
PER_CLASS_PATH = RESULTS_DIR / "test_per_class_accuracy_final.csv"
CONFUSIONS_PATH = RESULTS_DIR / "test_top_confusions_final.csv"
RAW_X_PATH = PACKED_DIR / "test_color_final_raw_X.npy"
META_PATH = PACKED_DIR / "test_color_final_metadata.csv"

ANALYSIS_DIR = RESULTS_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE, BASE.exists())
print("RESULTS_DIR:", RESULTS_DIR, RESULTS_DIR.exists())
print("ANALYSIS_DIR:", ANALYSIS_DIR)
print("RAW_X exists:", RAW_X_PATH.exists())

In [ ]:
# ============================================================
# 1) Test sonuçlarını yükle
# ============================================================
with open(METRICS_PATH, "r", encoding="utf-8") as f:
    metrics = json.load(f)

pred_df = pd.read_csv(PRED_PATH)
per_class_df = pd.read_csv(PER_CLASS_PATH)
conf_df = pd.read_csv(CONFUSIONS_PATH)

print("Metrics:")
print(json.dumps(metrics, indent=2, ensure_ascii=False))

print("\nPrediction rows:", len(pred_df))
print("Per-class rows:", len(per_class_df))
print("Confusion rows:", len(conf_df))

display(pred_df.head())
display(per_class_df.head())
display(conf_df.head())

In [ ]:
# ============================================================
# 2) Ana sonuç özeti
# ============================================================
summary = pd.DataFrame([{
    "test_samples": metrics["num_test_samples_used"],
    "num_classes": metrics["num_known_model_classes"],
    "top1_accuracy_%": metrics["top1_accuracy"] * 100,
    "top3_accuracy_%": metrics["top3_accuracy"] * 100,
    "top5_accuracy_%": metrics["top5_accuracy"] * 100,
    "mean_confidence_%": metrics["mean_confidence"] * 100,
    "preprocessing": metrics["preprocessing"],
    "ignored_files": metrics["ignored_files"],
}])

display(summary)

summary.to_csv(ANALYSIS_DIR / "summary_metrics.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 3) Confidence threshold analizi
# ============================================================
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

rows = []
for thr in thresholds:
    subset = pred_df[pred_df["confidence"] >= thr].copy()
    coverage = len(subset) / len(pred_df) if len(pred_df) > 0 else 0
    acc = subset["correct"].mean() if len(subset) > 0 else np.nan

    rows.append({
        "confidence_threshold": thr,
        "accepted_samples": len(subset),
        "coverage_%": coverage * 100,
        "accuracy_on_accepted_%": acc * 100 if not np.isnan(acc) else np.nan,
        "rejected_samples": len(pred_df) - len(subset),
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df)

threshold_df.to_csv(ANALYSIS_DIR / "confidence_threshold_analysis.csv", index=False, encoding="utf-8-sig")

print("Öneri:")
print("Demo için confidence >= 0.85 ise direkt tahmin, altındaysa Top-3 göster mantığı denenebilir.")

In [ ]:
# ============================================================
# 4) Doğru / yanlış confidence karşılaştırması
# ============================================================
conf_summary = (
    pred_df.groupby("correct")["confidence"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)

conf_summary["mean_%"] = conf_summary["mean"] * 100
conf_summary["median_%"] = conf_summary["median"] * 100

display(conf_summary)

conf_summary.to_csv(ANALYSIS_DIR / "correct_wrong_confidence_summary.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 5) En kötü class'lar
# ============================================================
worst_classes = per_class_df.sort_values(["accuracy", "n"], ascending=[True, False]).copy()
worst_classes["accuracy_%"] = worst_classes["accuracy"] * 100

display(worst_classes.head(30))

worst_classes.head(50).to_csv(ANALYSIS_DIR / "worst_50_classes.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 6) En iyi class'lar
# ============================================================
best_classes = per_class_df.sort_values(["accuracy", "n"], ascending=[False, False]).copy()
best_classes["accuracy_%"] = best_classes["accuracy"] * 100

display(best_classes.head(30))

best_classes.head(50).to_csv(ANALYSIS_DIR / "best_50_classes.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 7) En sık karışan class çiftleri
# ============================================================
top_confusions = conf_df.sort_values("count", ascending=False).copy()

display(top_confusions.head(50))

top_confusions.head(100).to_csv(ANALYSIS_DIR / "top_100_confusions.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 8) Yüksek confidence ama yanlış tahminler
#    Bunlar en kritik manuel kontrol adaylarıdır.
# ============================================================
high_conf_wrong = (
    pred_df[pred_df["correct"] == False]
    .sort_values("confidence", ascending=False)
    .copy()
)

display(high_conf_wrong.head(50))

high_conf_wrong.head(100).to_csv(ANALYSIS_DIR / "high_confidence_wrong_predictions.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 9) Düşük confidence ama doğru tahminler
#    Model doğru bilmiş ama kararsız kalmış.
# ============================================================
low_conf_correct = (
    pred_df[pred_df["correct"] == True]
    .sort_values("confidence", ascending=True)
    .copy()
)

display(low_conf_correct.head(50))

low_conf_correct.head(100).to_csv(ANALYSIS_DIR / "low_confidence_correct_predictions.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 10) Landmark kalite analizi
# ============================================================
# RAW_X shape beklenen: (N, 16, 126)
# 126 = left hand 63 + right hand 63

if not RAW_X_PATH.exists():
    raise FileNotFoundError(f"Raw landmark dosyası bulunamadı: {RAW_X_PATH}")

X_raw = np.load(RAW_X_PATH).astype(np.float32)

print("X_raw shape:", X_raw.shape)
assert X_raw.shape[0] == len(pred_df), "X_raw satır sayısı prediction CSV ile uyuşmuyor."
assert X_raw.shape[1:] == (16, 126), "X_raw shape beklenen (N,16,126) değil."

left = X_raw[:, :, :63]
right = X_raw[:, :, 63:]

left_frame_detected = np.any(np.abs(left) > 1e-8, axis=2)
right_frame_detected = np.any(np.abs(right) > 1e-8, axis=2)
any_frame_detected = left_frame_detected | right_frame_detected
both_frame_detected = left_frame_detected & right_frame_detected

quality_df = pred_df.copy()
quality_df["detected_frames"] = any_frame_detected.sum(axis=1)
quality_df["detected_frame_rate"] = quality_df["detected_frames"] / 16
quality_df["left_detected_frames"] = left_frame_detected.sum(axis=1)
quality_df["right_detected_frames"] = right_frame_detected.sum(axis=1)
quality_df["both_hands_detected_frames"] = both_frame_detected.sum(axis=1)
quality_df["both_hands_rate"] = quality_df["both_hands_detected_frames"] / 16
quality_df["raw_nonzero_count"] = np.count_nonzero(X_raw, axis=(1,2))

display(quality_df[[
    "sample_id",
    "true_original_class_id",
    "pred_original_class_id",
    "confidence",
    "correct",
    "detected_frames",
    "detected_frame_rate",
    "left_detected_frames",
    "right_detected_frames",
    "both_hands_detected_frames",
    "both_hands_rate",
]].head(20))

quality_df.to_csv(ANALYSIS_DIR / "prediction_with_landmark_quality.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 11) Landmark kalite özeti: doğru vs yanlış
# ============================================================
quality_summary = (
    quality_df.groupby("correct")
    .agg(
        n=("sample_id", "count"),
        mean_confidence=("confidence", "mean"),
        mean_detected_frames=("detected_frames", "mean"),
        mean_detected_frame_rate=("detected_frame_rate", "mean"),
        mean_left_frames=("left_detected_frames", "mean"),
        mean_right_frames=("right_detected_frames", "mean"),
        mean_both_hands_frames=("both_hands_detected_frames", "mean"),
        mean_both_hands_rate=("both_hands_rate", "mean"),
    )
    .reset_index()
)

display(quality_summary)

quality_summary.to_csv(ANALYSIS_DIR / "landmark_quality_correct_vs_wrong.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 12) Class bazlı landmark kalitesi
# ============================================================
class_quality = (
    quality_df.groupby("true_original_class_id")
    .agg(
        n=("sample_id", "count"),
        accuracy=("correct", "mean"),
        mean_confidence=("confidence", "mean"),
        mean_detected_frames=("detected_frames", "mean"),
        mean_detected_rate=("detected_frame_rate", "mean"),
        mean_both_hands_frames=("both_hands_detected_frames", "mean"),
        mean_both_hands_rate=("both_hands_rate", "mean"),
    )
    .reset_index()
)

# İsimleri per_class tablosundan ekle
name_cols = per_class_df[["true_original_class_id", "TR", "EN"]].drop_duplicates()
class_quality = class_quality.merge(name_cols, on="true_original_class_id", how="left")
class_quality["accuracy_%"] = class_quality["accuracy"] * 100
class_quality["mean_confidence_%"] = class_quality["mean_confidence"] * 100

# Hem accuracy düşük hem landmark kalitesi düşük olanlar
suspect_classes = class_quality.sort_values(
    ["accuracy", "mean_detected_rate", "mean_both_hands_rate"],
    ascending=[True, True, True]
)

display(suspect_classes.head(40))

suspect_classes.to_csv(ANALYSIS_DIR / "class_landmark_quality_analysis.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 13) Review listesi oluştur
#    Önce şu örnekleri manuel izlemek mantıklı:
#    - high confidence wrong
#    - worst class wrong examples
#    - low landmark quality wrong examples
# ============================================================
review_high_conf_wrong = high_conf_wrong.head(100).copy()
review_high_conf_wrong["review_reason"] = "high_confidence_wrong"

low_landmark_wrong = (
    quality_df[quality_df["correct"] == False]
    .sort_values(["detected_frame_rate", "confidence"], ascending=[True, False])
    .head(100)
    .copy()
)
low_landmark_wrong["review_reason"] = "low_landmark_quality_wrong"

worst_class_ids = worst_classes.head(20)["true_original_class_id"].tolist()
worst_class_wrong = (
    quality_df[
        (quality_df["true_original_class_id"].isin(worst_class_ids)) &
        (quality_df["correct"] == False)
    ]
    .sort_values(["true_original_class_id", "confidence"], ascending=[True, False])
    .copy()
)
worst_class_wrong["review_reason"] = "worst_class_wrong"

review_df = pd.concat([
    review_high_conf_wrong,
    low_landmark_wrong,
    worst_class_wrong,
], ignore_index=True)

review_df = review_df.drop_duplicates(subset=["sample_id", "review_reason"])

def color_video_path(sample_id):
    sid = str(sample_id)
    clean = sid.replace(".mp4", "").replace("_color", "").replace("_depth", "")
    return str(TEST_DIR / f"{clean}_color.mp4")

review_df["color_video_path"] = review_df["sample_id"].map(color_video_path)

review_cols = [
    "review_reason",
    "sample_id",
    "color_video_path",
    "true_original_class_id",
    "pred_original_class_id",
    "pred_TR",
    "pred_EN",
    "confidence",
    "correct",
    "detected_frames",
    "detected_frame_rate",
    "both_hands_detected_frames",
    "both_hands_rate",
]

existing_cols = [c for c in review_cols if c in review_df.columns]
review_df = review_df[existing_cols].copy()

review_path = ANALYSIS_DIR / "manual_review_video_list.csv"
review_df.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Review list kaydedildi:")
print(review_path)
print("Review sample count:", len(review_df))

display(review_df.head(50))

In [ ]:
# ============================================================
# 14) Belirli bir class için yanlış örnekleri listele
# ============================================================
def show_wrong_examples_for_class(class_id, n=10):
    class_id = int(class_id)

    sub = quality_df[
        (quality_df["true_original_class_id"] == class_id) &
        (quality_df["correct"] == False)
    ].sort_values("confidence", ascending=False).head(n).copy()

    if len(sub) == 0:
        print(f"Class {class_id} için yanlış örnek bulunamadı.")
        return sub

    sub["color_video_path"] = sub["sample_id"].map(color_video_path)

    cols = [
        "sample_id",
        "color_video_path",
        "true_original_class_id",
        "pred_original_class_id",
        "pred_TR",
        "pred_EN",
        "confidence",
        "detected_frames",
        "detected_frame_rate",
        "both_hands_detected_frames",
        "both_hands_rate",
        "top1_TR",
        "top1_prob",
        "top2_TR",
        "top2_prob",
        "top3_TR",
        "top3_prob",
    ]
    cols = [c for c in cols if c in sub.columns]

    display(sub[cols])
    return sub[cols]

# Örnek:
# show_wrong_examples_for_class(144, n=10)

In [ ]:
# ============================================================
# 15) Videoyu Colab içinde oynatmak için yardımcı fonksiyon
# ============================================================
def play_color_video(sample_id, width=420):
    sid = str(sample_id)
    path = Path(color_video_path(sid))

    print("Video:", path)
    print("Exists:", path.exists())

    if path.exists():
        display(Video(str(path), embed=True, width=width))
    else:
        print("Video bulunamadı.")

# Örnek:
# play_color_video("signer6_sample1")

In [ ]:
# ============================================================
# 16) Hızlı karar çıktıları
# ============================================================
print("Dosyalar kaydedildi:")
for p in sorted(ANALYSIS_DIR.glob("*")):
    print(" -", p.name)

print("\nÖnce incelenecek dosyalar:")
print("1)", ANALYSIS_DIR / "manual_review_video_list.csv")
print("2)", ANALYSIS_DIR / "worst_50_classes.csv")
print("3)", ANALYSIS_DIR / "top_100_confusions.csv")
print("4)", ANALYSIS_DIR / "prediction_with_landmark_quality.csv")
print("5)", ANALYSIS_DIR / "confidence_threshold_analysis.csv")